# Assignment 7: Creating and Fixing Mode Collapse in a DCGAN

## Objective

Use the working **DCGAN from Assignment 4** on the **Fashion-MNIST** dataset. In this experiment, the original model is deliberately made unstable to observe reduced diversity / mode-collapse-like behavior. Then a stabilization technique is applied and the results are compared.

**Based on Assignment 4**
- Framework: PyTorch
- Dataset: Fashion-MNIST
- Image size: 28 × 28 grayscale
- Latent dimension: 100
- Original learning rate: 0.0002
- Training length for this assignment: 15 epochs

### Experiment used in this notebook

**Broken model:** remove Batch Normalization from the Discriminator and use a much larger Discriminator learning rate.

**Improved model:** restore the original Discriminator architecture and use **different learning rates for Generator and Discriminator** (`G = 0.0002`, `D = 0.0001`) to make the adversarial training more balanced.


## 1. Import Libraries

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from PIL import Image

print("Libraries imported successfully!")


## 2. Reproducibility and Device

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)


## 3. Fashion-MNIST Dataset

Fashion-MNIST contains 28 × 28 grayscale clothing images. The images are normalized to `[-1, 1]` because the Generator uses `Tanh` at its output.


In [ ]:
BATCH_SIZE = 128
IMAGE_SIZE = 28
CHANNELS = 1
LATENT_DIM = 100
NUM_EPOCHS = 15

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("Training images:", len(train_dataset))
print("Number of batches:", len(train_loader))


## 4. Display Real Fashion-MNIST Samples

In [ ]:
class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

images, labels = next(iter(train_loader))

plt.figure(figsize=(10, 5))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    image = images[i].squeeze().numpy()
    image = (image + 1) / 2
    plt.imshow(image, cmap="gray")
    plt.title(class_names[labels[i].item()])
    plt.axis("off")

plt.tight_layout()
plt.show()


## 5. Original Assignment 4 Generator

The Generator follows the same architecture used in Assignment 4:

`100 × 1 × 1 → 128 × 7 × 7 → 64 × 14 × 14 → 1 × 28 × 28`


In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()

        self.model = nn.Sequential(
            nn.ConvTranspose2d(
                latent_dim, 128,
                kernel_size=7, stride=1, padding=0, bias=False
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(
                128, 64,
                kernel_size=4, stride=2, padding=1, bias=False
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(
                64, 1,
                kernel_size=4, stride=2, padding=1, bias=False
            ),
            nn.Tanh()
        )

    def forward(self, z):
        return self.model(z)


## 6. Broken Discriminator

To deliberately make the GAN unstable, Batch Normalization is removed from the Discriminator and the Discriminator is given a much larger learning rate.

This is different from simply changing the Generator capacity. The Generator architecture is kept the same as Assignment 4.


In [ ]:
class BrokenDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Conv2d(
                1, 64,
                kernel_size=4, stride=2, padding=1, bias=False
            ),
            nn.LeakyReLU(0.2, inplace=True),

            # Batch Normalization deliberately removed
            nn.Conv2d(
                64, 128,
                kernel_size=4, stride=2, padding=1, bias=False
            ),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(
                128, 1,
                kernel_size=7, stride=1, padding=0, bias=False
            ),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x).view(-1)


## 7. Improved Discriminator

For the improved model, the original Assignment 4 Discriminator is restored, including Batch Normalization.


In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Conv2d(
                1, 64,
                kernel_size=4, stride=2, padding=1, bias=False
            ),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(
                64, 128,
                kernel_size=4, stride=2, padding=1, bias=False
            ),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(
                128, 1,
                kernel_size=7, stride=1, padding=0, bias=False
            ),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x).view(-1)


## 8. DCGAN Weight Initialization

In [ ]:
def weights_init(model):
    classname = model.__class__.__name__

    if classname.find("Conv") != -1:
        nn.init.normal_(model.weight.data, 0.0, 0.02)

    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(model.weight.data, 1.0, 0.02)
        nn.init.constant_(model.bias.data, 0)

print("Weight initialization function ready.")


## 9. Fixed Noise

The same fixed noise vectors are used for both experiments. This makes the broken and improved outputs easier to compare because both Generators receive the same starting latent vectors.


In [ ]:
fixed_noise = torch.randn(
    16, LATENT_DIM, 1, 1, device=device
)

print("Fixed noise shape:", fixed_noise.shape)


# Part A — Deliberately Unstable GAN

## 10. Train the Broken Model

### Changes from Assignment 4
1. Batch Normalization is removed from the Discriminator.
2. The Discriminator learning rate is increased from `0.0002` to `0.001`.
3. The Generator keeps the original architecture and learning rate.

These changes intentionally disturb the balance between the Generator and Discriminator.


In [ ]:
def train_gan(netG, netD, optimizerG, optimizerD, epochs, run_name):
    criterion = nn.BCELoss()

    g_epoch_losses = []
    d_epoch_losses = []
    saved_epochs = []

    output_dir = f"assignment7_outputs/{run_name}"
    os.makedirs(output_dir, exist_ok=True)

    for epoch in range(1, epochs + 1):
        netG.train()
        netD.train()

        total_G_loss = 0.0
        total_D_loss = 0.0
        batches = 0

        for real_images, _ in train_loader:
            real_images = real_images.to(device)
            batch_size = real_images.size(0)

            real_labels = torch.ones(batch_size, device=device)
            fake_labels = torch.zeros(batch_size, device=device)

            # -------------------------
            # Train Discriminator
            # -------------------------
            netD.zero_grad()

            output_real = netD(real_images)
            loss_D_real = criterion(output_real, real_labels)

            noise = torch.randn(
                batch_size, LATENT_DIM, 1, 1, device=device
            )
            fake_images = netG(noise)

            output_fake = netD(fake_images.detach())
            loss_D_fake = criterion(output_fake, fake_labels)

            loss_D = loss_D_real + loss_D_fake
            loss_D.backward()
            optimizerD.step()

            # -------------------------
            # Train Generator
            # -------------------------
            netG.zero_grad()

            output_fake_for_G = netD(fake_images)
            loss_G = criterion(
                output_fake_for_G,
                real_labels
            )

            loss_G.backward()
            optimizerG.step()

            total_G_loss += loss_G.item()
            total_D_loss += loss_D.item()
            batches += 1

        avg_G = total_G_loss / batches
        avg_D = total_D_loss / batches

        g_epoch_losses.append(avg_G)
        d_epoch_losses.append(avg_D)

        if epoch % 5 == 0 or epoch == 1:
            netG.eval()
            with torch.no_grad():
                samples = netG(fixed_noise).cpu()

            grid = make_grid(samples)
            grid_path = os.path.join(
                output_dir, f"broken_epoch_{epoch:02d}.png"
            )
            plt.imsave(grid_path, grid, cmap="gray", vmin=0, vmax=1)
            saved_epochs.append(epoch)

        print(
            f"Epoch [{epoch:02d}/{epochs}] | "
            f"G Loss: {avg_G:.4f} | D Loss: {avg_D:.4f}"
        )

    return g_epoch_losses, d_epoch_losses, saved_epochs


In [ ]:
def make_grid(images, rows=4, cols=4):
    images = (images + 1) / 2
    images = images.squeeze(1).numpy()

    row_images = []
    for r in range(rows):
        row_images.append(
            np.concatenate(
                [images[r * cols + c] for c in range(cols)],
                axis=1
            )
        )

    return np.concatenate(row_images, axis=0)


In [ ]:
# Build the deliberately unstable model
netG_bad = Generator(LATENT_DIM).to(device)
netD_bad = BrokenDiscriminator().to(device)

netG_bad.apply(weights_init)
netD_bad.apply(weights_init)

BAD_G_LR = 0.0002
BAD_D_LR = 0.001

optimizerG_bad = optim.Adam(
    netG_bad.parameters(),
    lr=BAD_G_LR,
    betas=(0.5, 0.999)
)

optimizerD_bad = optim.Adam(
    netD_bad.parameters(),
    lr=BAD_D_LR,
    betas=(0.5, 0.999)
)

print("Broken model created.")
print("Generator LR:", BAD_G_LR)
print("Discriminator LR:", BAD_D_LR)

bad_G_losses, bad_D_losses, bad_epochs = train_gan(
    netG_bad,
    netD_bad,
    optimizerG_bad,
    optimizerD_bad,
    NUM_EPOCHS,
    "broken"
)


## 11. Broken / Collapsed Output

In [ ]:
netG_bad.eval()

with torch.no_grad():
    broken_images = netG_bad(fixed_noise).cpu()

broken_grid = make_grid(broken_images)

plt.figure(figsize=(8, 8))
plt.imshow(broken_grid, cmap="gray", vmin=0, vmax=1)
plt.title("Broken GAN - Final Output")
plt.axis("off")
plt.tight_layout()

os.makedirs("assignment7_outputs", exist_ok=True)
plt.savefig(
    "assignment7_outputs/broken_final_output.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()


### Observation of the Broken Model

The unstable model should be checked for:
- repeated or very similar clothing shapes,
- blurry or low-quality samples,
- loss instability,
- reduced diversity across the 16 generated images.

The exact severity of mode collapse can vary because GAN training is stochastic.


# Part B — Fixing the Training Problem

## 12. Improvement Technique: Different Learning Rates

The improved experiment uses **different learning rates for the Generator and Discriminator**.

- Generator learning rate: `0.0002`
- Discriminator learning rate: `0.0001`

The original Discriminator architecture, including Batch Normalization, is also restored. This gives the two networks a more balanced training process.


In [ ]:
# Build the improved model
netG_good = Generator(LATENT_DIM).to(device)
netD_good = Discriminator().to(device)

netG_good.apply(weights_init)
netD_good.apply(weights_init)

GOOD_G_LR = 0.0002
GOOD_D_LR = 0.0001

optimizerG_good = optim.Adam(
    netG_good.parameters(),
    lr=GOOD_G_LR,
    betas=(0.5, 0.999)
)

optimizerD_good = optim.Adam(
    netD_good.parameters(),
    lr=GOOD_D_LR,
    betas=(0.5, 0.999)
)

print("Improved model created.")
print("Generator LR:", GOOD_G_LR)
print("Discriminator LR:", GOOD_D_LR)

good_G_losses, good_D_losses, good_epochs = train_gan(
    netG_good,
    netD_good,
    optimizerG_good,
    optimizerD_good,
    NUM_EPOCHS,
    "improved"
)


## 13. Improved Output

In [ ]:
netG_good.eval()

with torch.no_grad():
    improved_images = netG_good(fixed_noise).cpu()

improved_grid = make_grid(improved_images)

plt.figure(figsize=(8, 8))
plt.imshow(improved_grid, cmap="gray", vmin=0, vmax=1)
plt.title("Improved GAN - Final Output")
plt.axis("off")
plt.tight_layout()

plt.savefig(
    "assignment7_outputs/improved_final_output.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()


## 14. Broken vs Improved Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(broken_grid, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Broken / Unstable GAN")
axes[0].axis("off")

axes[1].imshow(improved_grid, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Improved GAN")
axes[1].axis("off")

plt.tight_layout()
plt.savefig(
    "assignment7_outputs/broken_vs_improved.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()


## 15. Loss Comparison

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    range(1, NUM_EPOCHS + 1),
    bad_G_losses,
    label="Broken Generator"
)
plt.plot(
    range(1, NUM_EPOCHS + 1),
    bad_D_losses,
    label="Broken Discriminator"
)
plt.plot(
    range(1, NUM_EPOCHS + 1),
    good_G_losses,
    label="Improved Generator"
)
plt.plot(
    range(1, NUM_EPOCHS + 1),
    good_D_losses,
    label="Improved Discriminator"
)

plt.xlabel("Epoch")
plt.ylabel("Average Loss")
plt.title("Assignment 7: Broken vs Improved GAN Losses")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "assignment7_outputs/loss_comparison.png",
    dpi=200
)
plt.show()


# 16. Written Explanation

### 1. What modification caused the model to perform poorly?

The Discriminator was deliberately made less stable by removing its Batch Normalization layer and increasing its learning rate from `0.0002` to `0.001`. This made the competition between the Generator and Discriminator unbalanced and could cause unstable learning and reduced output diversity.

### 2. What did the generated images look like?

The broken model produced lower-quality and less diverse Fashion-MNIST-like images. Some samples may look similar to each other, which is an indication of reduced diversity or mode-collapse-like behavior.

### 3. Which technique did you use to improve the model?

I used **different learning rates for the Generator and Discriminator**. The Generator used a learning rate of `0.0002`, while the Discriminator used `0.0001`. I also restored the original Assignment 4 Discriminator architecture with Batch Normalization.

### 4. Did the solution improve the output? Why?

The improved setup was more balanced because the Discriminator was no longer learning much faster than the Generator. This can give the Generator more useful gradients and help it learn a wider variety of Fashion-MNIST patterns. The actual improvement should be judged from the generated comparison image and loss graph.

> **Important:** GAN training is stochastic. If the broken run does not show obvious severe mode collapse, describe the actual output honestly as reduced diversity / unstable training rather than claiming complete collapse.


# 17. Submission Files

After running the notebook, submit screenshots of:

1. **Broken / collapsed output:** `assignment7_outputs/broken_final_output.png`
2. **Improved output:** `assignment7_outputs/improved_final_output.png`
3. **Broken vs improved comparison:** `assignment7_outputs/broken_vs_improved.png`
4. **Loss comparison:** `assignment7_outputs/loss_comparison.png`

The notebook uses the same Fashion-MNIST DCGAN foundation as Assignment 4 but uses a different instability modification and improvement technique from the example assignment.
